# Arbitrary waveform send/capture with alignment

This notebook sends an arbitrary waveform to a selected Red Pitaya output channel, captures from a selected input channel, then aligns the original input waveform to the measured output using cross-correlation and `np.roll`.

Acquisition settings used here:
- Trigger source: `CH1_PE`
- Trigger delay: `8192` samples
- Decimation: `1`
- Capture size: `16384` samples

Because the trigger source is fixed to `CH1_PE`, IN1 must see a rising edge above the trigger level. If you output on OUT2 and only connect OUT2 to IN2, acquisition may not trigger unless IN1 also receives a trigger/sync signal.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import redpitaya_scpi as scpi


## User settings

Change these values for your experiment. `OUTPUT_CHANNEL` selects `OUT1` or `OUT2`; `INPUT_CHANNEL` selects `IN1` or `IN2` for data readback.

In [ ]:
IP = "169.254.77.151"

OUTPUT_CHANNEL = 1
INPUT_CHANNEL = 1

DAC_SAMPLE_RATE = 125_000_000
ADC_SAMPLE_RATE = 125_000_000
WAVEFORM_SAMPLES = 16_384
ACQ_SAMPLES = 16_384
DECIMATION = 1
SAMPLE_RATE = ADC_SAMPLE_RATE / DECIMATION

# This frequency makes one AWG point correspond to one 8 ns DAC sample.
FREQUENCY = DAC_SAMPLE_RATE / WAVEFORM_SAMPLES

AMPLITUDE = 0.7
TRIGGER_LEVEL = 0.1
TRIGGER_SOURCE = "CH1_PE"
TRIGGER_DELAY_SAMPLES = 8192
ACQ_TIMEOUT_SECONDS = 10

CAPTURE_DIR = Path("notebook_captures")


## Create or replace the arbitrary waveform

Replace the `waveform = ...` line with your own dataset. Values should normally stay in `[-1, +1]`; the actual voltage is controlled by `AMPLITUDE`.

In [ ]:
t = np.arange(WAVEFORM_SAMPLES) / DAC_SAMPLE_RATE

# Example arbitrary waveform: a smooth multi-tone signal with a sync-like rising edge at the start.
waveform = 0.35 * np.sin(2 * np.pi * 120_000 * t) + 0.20 * np.sin(2 * np.pi * 850_000 * t)
waveform[:32] = -0.8
waveform[32:96] = 0.8
waveform = np.clip(waveform, -1.0, 1.0)

plt.figure(figsize=(10, 3))
plt.plot(t * 1e6, waveform * AMPLITUDE)
plt.title("Original arbitrary waveform")
plt.xlabel("Time (us)")
plt.ylabel("Voltage (V)")
plt.grid(True)
plt.tight_layout()


## Helper functions

The functions below are deliberately commented. They keep the SCPI operations readable and make the timing/alignment assumptions explicit.

In [ ]:
def waveform_csv(values: np.ndarray) -> str:
    """Convert a NumPy waveform into the comma-separated format expected by SCPI."""
    # Red Pitaya accepts arbitrary waveform data as text values separated by commas.
    # Keeping 5 decimal places is usually enough for the AWG voltage resolution and
    # avoids sending unnecessarily huge text commands.
    return ",".join(f"{value:.5f}" for value in values)


def wait_complete(rp: scpi.scpi) -> None:
    """Ask Red Pitaya to report when previous SCPI commands are complete."""
    # *OPC? returns only after previous operations have completed according to the
    # SCPI server. This is useful after waveform upload/configuration commands.
    rp.txrx_txt("*OPC?")


def configure_generator(rp: scpi.scpi, chan: int, values: np.ndarray) -> None:
    """Load an arbitrary waveform onto OUT1 or OUT2 and prepare burst triggering."""
    # chan selects the Red Pitaya generator output: 1 for OUT1, 2 for OUT2.
    # The waveform is uploaded once, then a single burst is emitted when we send
    # SOUR{chan}:TRig:INT.
    rp.tx_txt(f"SOUR{chan}:FUNC ARBITRARY")
    rp.tx_txt(f"SOUR{chan}:TRAC:DATA:DATA " + waveform_csv(values))
    rp.tx_txt(f"SOUR{chan}:FREQ:FIX {FREQUENCY}")
    rp.tx_txt(f"SOUR{chan}:VOLT {AMPLITUDE}")
    rp.tx_txt(f"SOUR{chan}:BURS:STAT BURST")
    rp.tx_txt(f"SOUR{chan}:BURS:NCYC 1")
    rp.tx_txt(f"SOUR{chan}:BURS:NOR 1")
    rp.tx_txt(f"SOUR{chan}:INITValue 0")
    rp.tx_txt(f"SOUR{chan}:BURS:LASTValue 0")
    rp.tx_txt(f"SOUR{chan}:TRig:SOUR INT")


def configure_acquisition(rp: scpi.scpi) -> None:
    """Arm the ADC acquisition using CH1_PE and an 8192-sample trigger delay."""
    # ACQ:TRig:DLY controls where the trigger appears inside the acquisition buffer.
    # A positive 8192-sample delay asks the board to collect more useful samples
    # after the trigger instead of returning a mostly pre-trigger buffer.
    rp.tx_txt("ACQ:RST")
    rp.tx_txt(f"ACQ:DEC:Factor {DECIMATION}")
    rp.tx_txt("ACQ:DATA:Units VOLTS")
    rp.tx_txt("ACQ:DATA:FORMAT ASCII")
    rp.tx_txt(f"ACQ:TRig:LEV {TRIGGER_LEVEL}")
    rp.tx_txt(f"ACQ:TRig:DLY {TRIGGER_DELAY_SAMPLES}")
    rp.tx_txt("ACQ:START")
    rp.tx_txt(f"ACQ:TRig {TRIGGER_SOURCE}")

    # Give the board a moment to arm. If it has already triggered, the input/noise
    # crossed the trigger level before the waveform was sent.
    time.sleep(0.01)
    if rp.txrx_txt("ACQ:TRig:STAT?") == "TD":
        raise RuntimeError("Acquisition triggered early. Raise TRIGGER_LEVEL or check the CH1 input.")


def wait_for_acquisition(rp: scpi.scpi, timeout_s: float = ACQ_TIMEOUT_SECONDS) -> None:
    """Wait until the trigger has fired and the acquisition buffer is ready."""
    # First wait for trigger-detected state TD.
    # Then wait for ACQ:TRig:FILL? = 1, meaning the post-trigger samples are ready.
    deadline = time.perf_counter() + timeout_s
    while time.perf_counter() < deadline:
        if rp.txrx_txt("ACQ:TRig:STAT?") == "TD":
            break
        time.sleep(0.001)
    else:
        raise TimeoutError("Acquisition did not trigger before timeout.")

    while time.perf_counter() < deadline:
        if rp.txrx_txt("ACQ:TRig:FILL?") == "1":
            return
        time.sleep(0.001)

    raise TimeoutError("Acquisition buffer did not fill before timeout.")


def read_acquisition_channel(rp: scpi.scpi, chan: int, samples: int = ACQ_SAMPLES) -> np.ndarray:
    """Read captured samples from IN1 or IN2 immediately after the data query."""
    # This intentionally reads immediately after sending the data query. It avoids
    # mixing other SCPI queries into the socket response order.
    rp.tx_txt(f"ACQ:SOUR{chan}:DATA:TRig? {samples},POST_TRIG")
    raw = rp.rx_txt()
    return np.array(raw.strip("{}\n\r").replace("  ", "").split(","), dtype=np.float64)


def align_input_to_output(input_waveform: np.ndarray, output_capture: np.ndarray) -> tuple[np.ndarray, int]:
    """Roll the input waveform so its shape best lines up with the measured output."""
    # We use cross-correlation to estimate the sample lag between the intended input
    # and measured output. The returned aligned input has the same length as output.
    # Positive lag means the output appears later, so the input is rolled right.
    input_v = input_waveform * AMPLITUDE
    input_padded = np.resize(input_v, len(output_capture))
    x = input_padded - np.mean(input_padded)
    y = output_capture - np.mean(output_capture)
    corr = np.correlate(y, x, mode="same")
    lag = int(np.argmax(corr) - len(output_capture) // 2)
    aligned_input = np.roll(input_padded, lag)
    return aligned_input, lag


## Run the send/capture cycle

Wire the selected output to the selected input. Also remember that the trigger source is fixed to `CH1_PE`, so IN1 must see the trigger edge.

In [ ]:
rp = scpi.scpi(IP, timeout=15)

try:
    rp.tx_txt("GEN:RST")
    configure_generator(rp, OUTPUT_CHANNEL, waveform)
    rp.tx_txt(f"OUTPUT{OUTPUT_CHANNEL}:STATE ON")
    wait_complete(rp)

    configure_acquisition(rp)

    start = time.perf_counter()
    rp.tx_txt(f"SOUR{OUTPUT_CHANNEL}:TRig:INT")
    time.sleep((WAVEFORM_SAMPLES / DAC_SAMPLE_RATE) + 0.005)
    wait_for_acquisition(rp)
    captured = read_acquisition_channel(rp, INPUT_CHANNEL)
    elapsed = time.perf_counter() - start

finally:
    rp.close()

aligned_input, lag_samples = align_input_to_output(waveform, captured)
lag_us = lag_samples / SAMPLE_RATE * 1e6

print(f"Capture elapsed from trigger command to read complete: {elapsed * 1e3:.3f} ms")
print(f"Estimated alignment lag: {lag_samples} samples ({lag_us:.3f} us)")
print(f"Captured min/max: {np.min(captured):.4f} V / {np.max(captured):.4f} V")


## Save and plot results

The CSV file stores the aligned input and measured output with the same sample index. This is the useful training-pair representation.

In [ ]:
CAPTURE_DIR.mkdir(exist_ok=True)

time_us = np.arange(len(captured)) / SAMPLE_RATE * 1e6
np.savetxt(
    CAPTURE_DIR / "aligned_input_output.csv",
    np.column_stack((time_us, aligned_input, captured)),
    delimiter=",",
    header="time_us,aligned_input_v,captured_output_v",
    comments="",
)

plt.figure(figsize=(11, 4))
plt.plot(time_us, captured, label="Captured output")
plt.plot(time_us, aligned_input, "--", label="Rolled/aligned input")
plt.xlabel("Time (us)")
plt.ylabel("Voltage (V)")
plt.title("Aligned input vs captured output")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Saved: {CAPTURE_DIR / 'aligned_input_output.csv'}")
